# ⚙️ Feature Engineering

This notebook creates meaningful features from the cleaned dataset.

Features created:
- Time-based features
- Distance-based features
- Coordinate difference features
- Rush hour and weekend indicators
- Encoded categorical variables

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/train_clean.csv")

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1447928, 11)


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [3]:
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_day"] = df["pickup_datetime"].dt.day
df["pickup_month"] = df["pickup_datetime"].dt.month
df["pickup_weekday"] = df["pickup_datetime"].dt.weekday
df["is_weekend"] = df["pickup_weekday"].isin([5, 6]).astype(int)

In [4]:
df["is_rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
df["is_night"] = df["pickup_hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)

In [5]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [6]:
df["haversine_distance_km"] = haversine_distance(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

In [7]:
df["manhattan_distance_km"] = (
    haversine_distance(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["pickup_latitude"],
        df["dropoff_longitude"]
    )
    +
    haversine_distance(
        df["pickup_latitude"],
        df["pickup_longitude"],
        df["dropoff_latitude"],
        df["pickup_longitude"]
    )
)

In [8]:
df["longitude_difference"] = df["dropoff_longitude"] - df["pickup_longitude"]
df["latitude_difference"] = df["dropoff_latitude"] - df["pickup_latitude"]

In [9]:
def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    diff_lon = np.radians(lon2 - lon1)
    
    x = np.sin(diff_lon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (
        np.sin(lat1) * np.cos(lat2) * np.cos(diff_lon)
    )
    
    bearing = np.degrees(np.arctan2(x, y))
    
    return (bearing + 360) % 360

In [10]:
df["bearing"] = calculate_bearing(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["dropoff_latitude"],
    df["dropoff_longitude"]
)

In [11]:
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({"N": 0, "Y": 1})

In [12]:
df_model = df.drop(
    columns=[
        "id",
        "pickup_datetime",
        "dropoff_datetime"
    ]
)

df_model.head()

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day,pickup_month,pickup_weekday,is_weekend,is_rush_hour,is_night,haversine_distance_km,manhattan_distance_km,longitude_difference,latitude_difference,bearing
0,2,1,-73.982155,40.767937,-73.964630,40.765602,0,455,17,14,3,0,0,1,0,1.498521,1.735433,0.017525,-0.002335,99.970196
1,1,1,-73.980415,40.738564,-73.999481,40.731152,0,663,0,12,6,6,1,0,1,1.805507,2.430506,-0.019066,-0.007412,242.846232
2,2,1,-73.979027,40.763939,-74.005333,40.710087,0,2124,11,19,1,1,0,0,0,6.385098,8.203575,-0.026306,-0.053852,200.319835
3,2,1,-74.010040,40.719971,-74.012268,40.706718,0,429,19,6,4,2,0,1,0,1.485498,1.661331,-0.002228,-0.013252,187.262300
4,2,1,-73.973053,40.793209,-73.972923,40.782520,0,435,13,26,3,5,1,0,0,1.188588,1.199457,0.000130,-0.010689,179.473585


In [13]:
print("Final Shape:", df_model.shape)
df_model.info()

Final Shape: (1447928, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447928 entries, 0 to 1447927
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   vendor_id              1447928 non-null  int64  
 1   passenger_count        1447928 non-null  int64  
 2   pickup_longitude       1447928 non-null  float64
 3   pickup_latitude        1447928 non-null  float64
 4   dropoff_longitude      1447928 non-null  float64
 5   dropoff_latitude       1447928 non-null  float64
 6   store_and_fwd_flag     1447928 non-null  int64  
 7   trip_duration          1447928 non-null  int64  
 8   pickup_hour            1447928 non-null  int32  
 9   pickup_day             1447928 non-null  int32  
 10  pickup_month           1447928 non-null  int32  
 11  pickup_weekday         1447928 non-null  int32  
 12  is_weekend             1447928 non-null  int64  
 13  is_rush_hour           1447928 non-null  int6